In [1]:
import pandas as pd

# 1. Load Data (gunakan data asli atau data yang sudah dibersihkan sebelumnya)
file_path = 'scopus_export_Jan 17-2026.csv'
df = pd.read_csv(file_path)

# --- Ulangi Proses Pembersihan (jika memuat dari data mentah) ---
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(r'[^\w\s]', '', regex=True)
df = df.dropna(how='all')

text_cols = ['authors', 'author_full_names', 'title', 'source_title', 'abstract', 
             'author_keywords', 'index_keywords', 'publisher']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()

cols_to_fill = ['volume', 'issue', 'art_no', 'page_start', 'page_end', 'link']
for col in cols_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()

if 'year' in df.columns:
    df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)

df = df.drop_duplicates(subset=['title', 'year'])
df = df.reset_index(drop=True)
# -------------------------------------------------------------

# 2. Simpan ke Excel
output_excel = 'cleaned_scopus_data.xlsx'
df.to_excel(output_excel, index=False)

print(f"Data berhasil dikonversi ke: {output_excel}")

Data berhasil dikonversi ke: cleaned_scopus_data.xlsx


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random

def scrape_scopus_info(url):
    """
    Fungsi untuk mengambil data DOI dan Source Title dari link Scopus
    berdasarkan struktur HTML yang diberikan.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            data = {'doi': '', 'source_title': ''}
            
            # 1. Ambil DOI
            # Mencari span dengan data-testid="publication-doi"
            doi_tag = soup.find('span', {'data-testid': 'publication-doi'})
            if doi_tag:
                # Mengambil teks dan menghilangkan "DOI: " di depannya
                data['doi'] = doi_tag.get_text(strip=True).replace('DOI:', '').strip()
            
            # 2. Ambil Source Title (yang biasanya jadi Publisher/Journal Name)
            # Mencari div wrapper info publikasi
            info_bar = soup.find('div', {'data-testid': 'publication-information-bar'})
            if info_bar:
                # Biasanya judul jurnal/prosiding ada di span pertama di dalam div tersebut
                # Class sesuai snippet: Typography_typography__CrPwo...
                title_span = info_bar.find('span') 
                if title_span:
                     data['source_title'] = title_span.get_text(strip=True)
            
            return data
            
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

    return None

# --- PROGRAM UTAMA ---

# 1. Load Data Excel
filename = 'cleaned_scopus_data.xlsx'
print(f"Membaca file {filename}...")
df = pd.read_excel(filename)

# Pastikan kolom DOI ada
if 'doi' not in df.columns:
    df['doi'] = ''

# 2. Loop untuk baris yang publisher/source_title-nya kosong
#    (Kita asumsikan jika source_title kosong, kita perlu scrape)
print("Memulai proses scraping untuk data yang kosong...")

count = 0
for index, row in df.iterrows():
    # Cek kondisi: Link ada, tapi Source Title atau Publisher kosong (atau DOI kosong)
    # Sesuaikan logika ini dengan kebutuhan Anda. Di sini saya cek jika source_title kosong ATAU doi kosong.
    if pd.notna(row['link']) and (pd.isna(row['source_title']) or row['source_title'] == '' or pd.isna(row['doi']) or row['doi'] == ''):
        
        url = row['link']
        print(f"Processing row {index}: {url}")
        
        scraped_data = scrape_scopus_info(url)
        
        if scraped_data:
            # Update DOI jika ditemukan
            if scraped_data['doi']:
                df.at[index, 'doi'] = scraped_data['doi']
                print(f"   -> Found DOI: {scraped_data['doi']}")
            
            # Update Source Title jika ditemukan dan yang lama kosong
            if scraped_data['source_title'] and (pd.isna(row['source_title']) or row['source_title'] == ''):
                df.at[index, 'source_title'] = scraped_data['source_title']
                # Opsional: Jika kolom publisher juga mau disamakan dengan source title
                if 'publisher' in df.columns and (pd.isna(row['publisher']) or row['publisher'] == ''):
                     df.at[index, 'publisher'] = scraped_data['source_title']
                print(f"   -> Found Source: {scraped_data['source_title']}")
                
            # Simpan setiap 10 data agar aman jika script putus
            if count % 10 == 0:
                df.to_excel('cleaned_scopus_data_updated.xlsx', index=False)
                
            count += 1
            # Beri jeda waktu agar tidak dianggap bot spam oleh Scopus
            time.sleep(random.uniform(2, 5))

# 3. Simpan Hasil Akhir
output_file = 'cleaned_scopus_data_bismillah.xlsx'
df.to_excel(output_file, index=False)
print(f"Selesai! Data disimpan ke {output_file}")

Membaca file cleaned_scopus_data.xlsx...
Memulai proses scraping untuk data yang kosong...
Processing row 0: https://www.scopus.com/inward/record.uri?eid=2-s2.0-105026394658&partnerID=40&md5=67356aa672dd1dff46fd1aa8d1602b75
Processing row 1: https://www.scopus.com/inward/record.uri?eid=2-s2.0-105025349593&partnerID=40&md5=aee06610d9c5168f3dc8da907c085a80
Processing row 2: https://www.scopus.com/inward/record.uri?eid=2-s2.0-105009598983&partnerID=40&md5=2e54e8317f900af6af80d1df3efabd7d
Processing row 3: https://www.scopus.com/inward/record.uri?eid=2-s2.0-105025708551&partnerID=40&md5=f5c36104dd350700f9e6e571056ea3e5
Processing row 4: https://www.scopus.com/inward/record.uri?eid=2-s2.0-105025692081&partnerID=40&md5=a10b46e10b93c94d0deaadd5a3e9a571
Processing row 5: https://www.scopus.com/inward/record.uri?eid=2-s2.0-105025361649&partnerID=40&md5=8dbf4058daa4d9a9bc6ce81794f1dc23
Processing row 6: https://www.scopus.com/inward/record.uri?eid=2-s2.0-105025010347&partnerID=40&md5=a519bb1f768

PermissionError: [Errno 13] Permission denied: 'cleaned_scopus_data_final.xlsx'

In [ ]:
import pandas as pd

# 1. Load data dari file Excel yang sebelumnya dibuat (sebagai data baru)
df = pd.read_excel('cleaned_scopus_data.xlsx')

# 2. Daftar kolom yang akan dihapus
# (Nama kolom disesuaikan dengan format standar yang sudah dilakukan sebelumnya: huruf kecil & underscore)
cols_to_drop = [
    'author_full_names', 
    'authors_id', 
    'volume', 
    'issue', 
    'art_no', 
    'page_start', 
    'page_end', 
    'abbreviated_source_title'
]

# 3. Hapus kolom jika ada di dataframe
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

# 4. Tampilkan Info dan Head untuk memastikan kolom sudah terhapus
print("Info Dataframe setelah penghapusan kolom:")
print(df.info())

print("\nContoh 5 baris pertama data:")
print(df.head())

Info Dataframe setelah penghapusan kolom:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18613 entries, 0 to 18612
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   authors          17717 non-null  object
 1   title            18613 non-null  object
 2   year             18613 non-null  int64 
 3   source_title     13021 non-null  object
 4   link             18613 non-null  object
 5   abstract         18613 non-null  object
 6   author_keywords  14653 non-null  object
 7   index_keywords   12579 non-null  object
 8   publisher        17315 non-null  object
dtypes: int64(1), object(8)
memory usage: 1.3+ MB
None

Contoh 5 baris pertama data:
                                             authors  \
0                   Shinyclimensa, C.; Parthiban, A.   
1                                    Shashikumar, N.   
2  Budhathoki, M.; Li, L.; Xu, H.; Zhang, W.; Ma,...   
3       Ullah, Q.; Qiu, Y.; Khan Kakar, S.; Sa

In [5]:
df.head()

,authors,title,year,source_title,link,abstract,author_keywords,index_keywords,publisher
0,"Shinyclimensa, C.; Parthiban, A.",Forecasting cashew production in India using a...,2026,Scientific Reports,https://www.scopus.com/inward/record.uri?eid=2...,This study presents a comprehensive analytical...,Cashew production forecasting; Centrality meas...,NaN,Nature Research
1,"Shashikumar, N.",Predictive maintenance and inventory optimizat...,2026,Network Modeling Analysis in Health Informatic...,https://www.scopus.com/inward/record.uri?eid=2...,"In today’s healthcare, making sure medical equ...",Inventory optimization; Medical device supply ...,Artificial intelligence; Biomedical equipment;...,Springer
2,"Budhathoki, M.; Li, L.; Xu, H.; Zhang, W.; Ma,...",Market dynamics and E-commerce satisfaction in...,2026,Aquaculture,https://www.scopus.com/inward/record.uri?eid=2...,"China's growing demand for aquatic foods, driv...",Consumer preferences; E-commerce satisfaction;...,competitiveness; electronic commerce; food mar...,Elsevier B.V.
3,"Ullah, Q.; Qiu, Y.; Khan Kakar, S.; Sami, M.",Revolutionizing European digital exports: The ...,2026,Technology in Society,https://www.scopus.com/inward/record.uri?eid=2...,Reconciling economic growth with sustainabilit...,Digital trade; Global supply chain; Green FinT...,Economic analysis; Fintech; Industrial relatio...,Elsevier Ltd
4,"Al-Ababneh, H.A.; Kafka, S.; Serbina, T.; Maib...",The impact of blockchain systems on the transp...,2026,Multidisciplinary Science Journal,https://www.scopus.com/inward/record.uri?eid=2...,The implementation of blockchain technologies ...,competitiveness; correlation; investment in AI...,NaN,Malque Publishing


In [6]:
# 3. Filter Tahun (Hanya simpan 2015 <= Tahun <= 2025)
df = df[(df['year'] >= 2015) & (df['year'] <= 2025)]

# Tampilkan Informasi Data Hasil Filter
print(f"Data tersisa: {len(df)} baris")
print("Tahun Min:", df['year'].min())
print("Tahun Max:", df['year'].max())
print(df.head())

Data tersisa: 16182 baris
Tahun Min: 2015
Tahun Max: 2025
                                               authors  \
396                        Wang, H.; Wang, L.; Zhu, F.   
397                               Pham, H.B.; Briš, P.   
398  Luo, H.; Akkermans, S.; Engels, W.; Jacobs, S....   
399                  Giri, P.; Debnath, B.K.; Paul, S.   
400                                  Lai, Y.; Yuan, Y.   

                                                 title  year  \
396  Enhancing Renewable Energy Forecasting Through...  2025   
397  AI in Supply Chain: Techniques, Applications, ...  2025   
398  Hybrid plant-dairy cheese: Effects of lactic a...  2025   
399  A hybrid fuzzy decision-making framework for m...  2025   
400  Comparative Analysis of BSS, PTO, and E2E Stra...  2025   

                                          source_title  \
396  International Journal of Renewable Energy Rese...   
397             Operations and Supply Chain Management   
398                               

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16182 entries, 396 to 16577
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   authors          15475 non-null  object
 1   title            16182 non-null  object
 2   year             16182 non-null  int64 
 3   source_title     11331 non-null  object
 4   link             16182 non-null  object
 5   abstract         16182 non-null  object
 6   author_keywords  13169 non-null  object
 7   index_keywords   10518 non-null  object
 8   publisher        16147 non-null  object
dtypes: int64(1), object(8)
memory usage: 1.2+ MB


In [11]:
df = df.drop_duplicates(subset=['title', 'abstract'], keep='first')

# Cek hasil
print(f"Jumlah data setelah hapus duplikat Title & Abstract: {len(df)}")

Jumlah data setelah hapus duplikat Title & Abstract: 16179


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6286 entries, 396 to 16574
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   authors          6286 non-null   object
 1   title            6286 non-null   object
 2   year             6286 non-null   int64 
 3   source_title     6286 non-null   object
 4   link             6286 non-null   object
 5   abstract         6286 non-null   object
 6   author_keywords  6286 non-null   object
 7   index_keywords   6286 non-null   object
 8   publisher        6286 non-null   object
dtypes: int64(1), object(8)
memory usage: 491.1+ KB


In [15]:
df.head()

,authors,title,year,source_title,link,abstract,author_keywords,index_keywords,publisher
396,"Wang, H.; Wang, L.; Zhu, F.",Enhancing Renewable Energy Forecasting Through...,2025,International Journal of Renewable Energy Rese...,https://www.scopus.com/inward/record.uri?eid=2...,Managing energy resources effectively involves...,Collaborative control mode; Energy supply chai...,Adaptive boosting; Balancing; Economics; Energ...,Gazi Universitesi
398,"Luo, H.; Akkermans, S.; Engels, W.; Jacobs, S....",Hybrid plant-dairy cheese: Effects of lactic a...,2025,Food Chemistry,https://www.scopus.com/inward/record.uri?eid=2...,In the search to improve the sustainability of...,Fermentation; GC–MS; Lactic acid bacteria; Mac...,Bacteria; Cheeses; Dairies; Food supply; Learn...,Elsevier Ltd
399,"Giri, P.; Debnath, B.K.; Paul, S.",A hybrid fuzzy decision-making framework for m...,2025,Engineering Applications of Artificial Intelli...,https://www.scopus.com/inward/record.uri?eid=2...,The impact of global warming has arisen as one...,Compromise Ranking of Alternatives from Distan...,Carbon; Decision making; Fuzzy logic; Fuzzy se...,Elsevier Ltd
408,"Suthagar, K.S.; Thomas, A.; Mishra, U.",Recycling aluminium and solar truck efficiency...,2025,Journal of Cleaner Production,https://www.scopus.com/inward/record.uri?eid=2...,A sustainable economic model is crucial in pre...,Carbon capture-storage; Forecasting model; Rec...,Aluminum; Carbon capture; Carbon capture and s...,Elsevier Ltd
409,"Zhu, J.; Han, W.; Dong, F.; Shi, L.",Factors influencing 3PLs’ willingness to adopt...,2025,Journal of Cleaner Production,https://www.scopus.com/inward/record.uri?eid=2...,Based on a systematic investigation into the o...,Climate change adaptation; Machine learning; P...,Additives; Artificial intelligence; Climate mo...,Elsevier Ltd


In [13]:
df.dropna(inplace=True)

C:\Users\TUF\AppData\Local\Temp\ipykernel_14584\1379821321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)
